In [ ]:
import os
import random
import glob
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from scipy import ndimage
import cv2
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import time
from tensorflow.keras.utils import Progbar
import math
from tqdm import tqdm
import csv

In [ ]:
# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = "/kaggle/input/isic-2018-image-and-mask-resized-256/ISIC_2018_Image_and_Mask"
IMG_DIR = os.path.join(DATA_DIR, "Images")
MASK_DIR = os.path.join(DATA_DIR, "Mask")
OUTPUT_DIR = "./lafr_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

INPUT_SIZE = (256, 256)
PRETRAIN_CROP = 128
PRETRAIN_BATCH = 32
PRETRAIN_EPOCHS = 1
BATCH = 8
FINETUNE_EPOCHS = 1
REFINER_EPOCHS = 1
LR = 1e-4

In [ ]:
# GPU memory growth (optional)
gpus = tf.config.list_physical_devices('GPU')
for g in gpus:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass

In [ ]:
# -------------------------
# Utilities: window partition / reverse
# -------------------------
def window_partition(x, window_size):
    B = tf.shape(x)[0]
    H = tf.shape(x)[1]
    W = tf.shape(x)[2]
    C = tf.shape(x)[3]
    x = tf.reshape(x, (B, H // window_size, window_size, W // window_size, window_size, C))
    x = tf.transpose(x, perm=[0,1,3,2,4,5])
    windows = tf.reshape(x, (-1, window_size, window_size, C))
    return windows

def window_reverse(windows, window_size, H, W, B):
    x = tf.reshape(windows, (B, H // window_size, W // window_size, window_size, window_size, -1))
    x = tf.transpose(x, perm=[0,1,3,2,4,5])
    x = tf.reshape(x, (B, H, W, -1))
    return x

In [ ]:
# -------------------------
# Serializable custom layers
# -------------------------
@tf.keras.utils.register_keras_serializable(package="lafr_custom")
class ResizeToTarget(layers.Layer):
    def __init__(self, method='bilinear', **kwargs):
        super().__init__(**kwargs)
        self.method = method

    def call(self, inputs):
        x, target = inputs
        target_shape = tf.shape(target)[1:3]
        return tf.image.resize(x, target_shape, method=self.method)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"method": self.method})
        return cfg


In [ ]:
@tf.keras.utils.register_keras_serializable(package="lafr_custom")
class WindowAttention(layers.Layer):
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, attn_drop=0., proj_drop=0., **kwargs):
        super().__init__(**kwargs)
        self.dim = int(dim)
        self.window_size = int(window_size)
        self.num_heads = int(num_heads)
        assert self.dim % self.num_heads == 0, "dim must be divisible by num_heads"
        self.head_dim = self.dim // self.num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv_bias = qkv_bias
        self.attn_drop_rate = attn_drop
        self.proj_drop_rate = proj_drop

        # Dense/proj declared here; build() will create weights with known dims
        self.qkv = layers.Dense(self.dim * 3)
        self.proj = layers.Dense(self.dim)
        self.attn_drop = layers.Dropout(self.attn_drop_rate)
        self.proj_drop = layers.Dropout(self.proj_drop_rate)

        # placeholder for relative table/index created in build()
        self.relative_position_bias_table = None
        self.relative_position_index = None

    def build(self, input_shape):
        ws = int(self.window_size)
        table_size = (2*ws-1)*(2*ws-1)
        self.relative_position_bias_table = self.add_weight(
            name='rel_pos_bias',
            shape=(table_size, self.num_heads),
            initializer='zeros',
            trainable=True
        )

        coords = np.stack(np.meshgrid(np.arange(ws), np.arange(ws), indexing='ij'))
        coords_flat = coords.reshape(2, -1)
        relative_coords = coords_flat[:, :, None] - coords_flat[:, None, :]
        relative_coords = relative_coords + (ws - 1)
        relative_coords = relative_coords[0] * (2*ws-1) + relative_coords[1]
        relative_index = relative_coords.reshape(-1).astype(np.int32)
        self.relative_position_index = tf.constant(relative_index)

        # ensure Dense layers are built
        self.qkv.build((None, self.dim))
        self.proj.build((None, self.dim))

        super().build(input_shape)

    def call(self, x, mask=None, training=None):
        Bn = tf.shape(x)[0]
        N = tf.shape(x)[1]
        qkv = self.qkv(x)
        qkv = tf.reshape(qkv, (Bn, N, 3, self.num_heads, self.head_dim))
        qkv = tf.transpose(qkv, perm=[2,0,3,1,4])
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = q * self.scale
        attn = tf.matmul(q, k, transpose_b=True)

        relative_bias = tf.gather(self.relative_position_bias_table, self.relative_position_index)
        relative_bias = tf.reshape(relative_bias, (N, N, -1))
        relative_bias = tf.transpose(relative_bias, perm=[2,0,1])
        attn = attn + tf.expand_dims(relative_bias, 0)

        if mask is not None:
            attn = attn + (mask * -10000.0)

        attn = tf.nn.softmax(attn, axis=-1)
        attn = self.attn_drop(attn, training=training)
        x = tf.matmul(attn, v)
        x = tf.transpose(x, perm=[0,2,1,3])
        x = tf.reshape(x, (Bn, N, self.dim))
        x = self.proj(x)
        x = self.proj_drop(x, training=training)
        return x

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "dim": self.dim,
            "window_size": self.window_size,
            "num_heads": self.num_heads,
            "qkv_bias": self.qkv_bias,
            "attn_drop": self.attn_drop_rate,
            "proj_drop": self.proj_drop_rate
        })
        return cfg

In [ ]:
@tf.keras.utils.register_keras_serializable(package="lafr_custom")
class MLP(layers.Layer):
    def __init__(self, in_features, hidden_features=None, drop=0., **kwargs):
        super().__init__(**kwargs)
        self.in_features = int(in_features)
        self.hidden_features = int(hidden_features or in_features)
        self.drop_rate = float(drop)

        self.fc1 = layers.Dense(self.hidden_features)
        self.act = layers.Activation('gelu')
        self.drop1 = layers.Dropout(self.drop_rate)
        self.fc2 = layers.Dense(self.in_features)
        self.drop2 = layers.Dropout(self.drop_rate)

    def call(self, x, training=None):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x, training=training)
        x = self.fc2(x)
        x = self.drop2(x, training=training)
        return x

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "in_features": self.in_features,
            "hidden_features": self.hidden_features,
            "drop": self.drop_rate
        })
        return cfg

In [ ]:
@tf.keras.utils.register_keras_serializable(package="lafr_custom")
class SwinTransformerBlock(layers.Layer):
    def __init__(self, dim, num_heads, window_size=7, shift_size=0,
                 mlp_ratio=4., qkv_bias=True, drop=0., attn_drop=0., drop_path=0., **kwargs):
        super().__init__(**kwargs)
        self.dim = int(dim)
        self.num_heads = int(num_heads)
        self.window_size = int(window_size)
        self.shift_size = int(shift_size) if shift_size < window_size else 0
        self.mlp_ratio = float(mlp_ratio)
        self.qkv_bias = qkv_bias
        self.drop = drop
        self.attn_drop = attn_drop
        self.drop_path_rate = drop_path

        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = WindowAttention(dim, window_size=window_size, num_heads=num_heads, qkv_bias=qkv_bias,
                                    attn_drop=attn_drop, proj_drop=drop)
        self.drop_path = layers.Dropout(self.drop_path_rate) if self.drop_path_rate>0. else layers.Activation('linear')
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = MLP(dim, int(dim*self.mlp_ratio), drop)

    def build(self, input_shape):
        # build child layers as needed
        self.norm1.build(input_shape)
        self.attn.build((None, self.window_size*self.window_size, self.dim))
        self.norm2.build(input_shape)
        self.mlp.build((None, None, None, self.dim))
        super().build(input_shape)

    def call(self, x, training=None):
        B = tf.shape(x)[0]
        H = tf.shape(x)[1]
        W = tf.shape(x)[2]
        C_static = x.shape[-1]

        shortcut = x
        x = self.norm1(x)

        pad_h = (self.window_size - H % self.window_size) % self.window_size
        pad_w = (self.window_size - W % self.window_size) % self.window_size
        x = tf.pad(x, [[0,0],[0,pad_h],[0,pad_w],[0,0]])
        Hp = tf.shape(x)[1]; Wp = tf.shape(x)[2]

        if self.shift_size > 0:
            x = tf.roll(x, shift=[-self.shift_size, -self.shift_size], axis=[1,2])

        x_windows = window_partition(x, self.window_size)
        N = self.window_size * self.window_size
        x_windows = tf.reshape(x_windows, (-1, N, C_static))

        attn_windows = self.attn(x_windows, mask=None, training=training)
        attn_windows = tf.reshape(attn_windows, (-1, self.window_size, self.window_size, C_static))
        x = window_reverse(attn_windows, self.window_size, Hp, Wp, B)

        if self.shift_size > 0:
            x = tf.roll(x, shift=[self.shift_size, self.shift_size], axis=[1,2])

        x = x[:, :H, :W, :]

        x = shortcut + self.drop_path(x, training=training)
        x = x + self.drop_path(self.mlp(self.norm2(x), training=training), training=training)
        return x

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "dim": self.dim,
            "num_heads": self.num_heads,
            "window_size": self.window_size,
            "shift_size": self.shift_size,
            "mlp_ratio": self.mlp_ratio,
            "qkv_bias": self.qkv_bias,
            "drop": self.drop,
            "attn_drop": self.attn_drop,
            "drop_path": self.drop_path_rate
        })
        return cfg

In [ ]:
@tf.keras.utils.register_keras_serializable(package="lafr_custom")
class PatchEmbed(layers.Layer):
    def __init__(self, patch_size=4, embed_dim=96, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = int(patch_size)
        self.embed_dim = int(embed_dim)
        self.proj = layers.Conv2D(self.embed_dim, kernel_size=self.patch_size, strides=self.patch_size, padding='same')
        self.norm = layers.LayerNormalization(epsilon=1e-6)

    def call(self, x):
        x = self.proj(x)
        x = self.norm(x)
        return x

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"patch_size": self.patch_size, "embed_dim": self.embed_dim})
        return cfg

In [ ]:
@tf.keras.utils.register_keras_serializable(package="lafr_custom")
class PatchMerging(layers.Layer):
    def __init__(self, out_dim, **kwargs):
        super().__init__(**kwargs)
        self.out_dim = int(out_dim)
        self.reduction = layers.Conv2D(self.out_dim, kernel_size=1, strides=1, padding="valid")
        self.norm = layers.LayerNormalization(epsilon=1e-6)

    def call(self, x):
        H = tf.shape(x)[1]
        W = tf.shape(x)[2]

        pad_h = tf.math.floormod(H, 2)
        pad_w = tf.math.floormod(W, 2)
        paddings = tf.cast(tf.stack([[0,0],[0,pad_h],[0,pad_w],[0,0]]), tf.int32)
        x = tf.pad(x, paddings)

        x = tf.nn.space_to_depth(x, block_size=2)
        x = self.reduction(x)
        x = self.norm(x)
        return x

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"out_dim": self.out_dim})
        return cfg

In [ ]:
@tf.keras.utils.register_keras_serializable(package="lafr_custom")
class BasicLayer(layers.Layer):
    def __init__(self, dim, depth, num_heads, window_size, mlp_ratio=4., downsample=None, name=None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.dim = int(dim)
        self.depth = int(depth)
        self.num_heads = int(num_heads)
        self.window_size = int(window_size)
        self.mlp_ratio = float(mlp_ratio)
        self.downsample = downsample
        self.blocks = []
        for i in range(self.depth):
            shift_size = 0 if (i % 2 == 0) else window_size // 2
            blk = SwinTransformerBlock(dim=dim, num_heads=num_heads, window_size=window_size, shift_size=shift_size, mlp_ratio=mlp_ratio)
            self.blocks.append(blk)
            setattr(self, f'block_{i}', blk)
        if downsample is not None:
            setattr(self, 'downsample_layer', downsample)

    def build(self, input_shape):
        for blk in self.blocks:
            blk.build(input_shape)
        if self.downsample is not None:
            self.downsample.build(input_shape)
        super().build(input_shape)

    def call(self, x, training=None):
        for blk in self.blocks:
            x = blk(x, training=training)
        if self.downsample is not None:
            x = self.downsample(x)
        return x

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "dim": self.dim,
            "depth": self.depth,
            "num_heads": self.num_heads,
            "window_size": self.window_size,
            "mlp_ratio": self.mlp_ratio,
            # downsample is not serializable if it's a layer; usually None here.
        })
        return cfg

In [ ]:
@tf.keras.utils.register_keras_serializable(package="lafr_custom")
class CBAM(layers.Layer):
    def __init__(self, ratio=8, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.ratio = int(ratio)
        self.kernel_size = int(kernel_size)

    def build(self, input_shape):
        ch = int(input_shape[-1])
        self.mlp1 = layers.Conv2D(max(ch // self.ratio, 1), 1, activation='relu')
        self.mlp2 = layers.Conv2D(ch, 1)
        self.spatial_conv = layers.Conv2D(1, self.kernel_size, padding='same', activation='sigmoid')
        super().build(input_shape)

    def call(self, x):
        avg_pool = tf.reduce_mean(x, axis=[1,2], keepdims=True)
        max_pool = tf.reduce_max(x, axis=[1,2], keepdims=True)

        a1 = self.mlp2(self.mlp1(avg_pool))
        a2 = self.mlp2(self.mlp1(max_pool))

        channel_attn = tf.nn.sigmoid(a1 + a2)
        x = x * channel_attn

        avg_sp = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_sp = tf.reduce_max(x, axis=-1, keepdims=True)
        sp = tf.concat([avg_sp, max_sp], axis=-1)
        sp = self.spatial_conv(sp)
        out = x * sp
        return out

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"ratio": self.ratio, "kernel_size": self.kernel_size})
        return cfg

In [ ]:
@tf.keras.utils.register_keras_serializable(package="lafr_custom")
class SEBlock(layers.Layer):
    def __init__(self, ratio=16, **kwargs):
        super().__init__(**kwargs)
        self.ratio = int(ratio)

    def build(self, input_shape):
        channels = int(input_shape[-1])
        self.global_pool = layers.GlobalAveragePooling2D()
        self.fc1 = layers.Dense(max(channels // self.ratio, 1), activation='relu')
        self.fc2 = layers.Dense(channels, activation='sigmoid')
        super().build(input_shape)

    def call(self, x):
        se = self.global_pool(x)
        se = self.fc1(se)
        se = self.fc2(se)
        se = tf.expand_dims(se, axis=1)
        se = tf.expand_dims(se, axis=1)
        return x * se

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"ratio": self.ratio})
        return cfg

In [ ]:
# -------------------------
# Build Swin-UNet (uses registered layers)
# -------------------------
def build_swin_unet_tf(input_shape=(256,256,3), out_channels=1,
                       embed_dim=96, depths=(2,2,6,2), num_heads=(3,6,12,24),
                       window_size=7, base_filters=32, use_cbam=True, use_se=False):
    inputs = layers.Input(shape=input_shape)
    x = PatchEmbed(patch_size=4, embed_dim=embed_dim)(inputs)   # /4

    x = BasicLayer(dim=embed_dim, depth=depths[0], num_heads=num_heads[0], window_size=window_size, name='basic_layer_1')(x)
    feat1 = x
    x = PatchMerging(out_dim=embed_dim*2)(x)
    x = BasicLayer(dim=embed_dim*2, depth=depths[1], num_heads=num_heads[1], window_size=window_size, name='basic_layer_2')(x)
    feat2 = x
    x = PatchMerging(out_dim=embed_dim*4)(x)
    x = BasicLayer(dim=embed_dim*4, depth=depths[2], num_heads=num_heads[2], window_size=window_size, name='basic_layer_3')(x)
    feat3 = x
    x = PatchMerging(out_dim=embed_dim*8)(x)
    x = BasicLayer(dim=embed_dim*8, depth=depths[3], num_heads=num_heads[3], window_size=window_size, name='basic_layer_4')(x)
    feat4 = x

    def proj(feat, out_ch):
        p = layers.Conv2D(out_ch, kernel_size=1, padding='same', activation='relu')(feat)
        if use_se:
            p = SEBlock(ratio=16)(p)
        return p

    p1 = proj(feat1, base_filters)
    p2 = proj(feat2, base_filters*2)
    p3 = proj(feat3, base_filters*4)
    p4 = proj(feat4, base_filters*8)

    def upsample_to(ref, target):
        return ResizeToTarget()([ref, target])

    d3 = layers.Concatenate(axis=-1)([p3, upsample_to(p4, p3)])
    d3 = layers.Conv2D(base_filters*4, 3, padding='same', activation='relu')(d3)
    if use_cbam: d3 = CBAM()(d3)

    d2 = layers.Concatenate(axis=-1)([p2, upsample_to(d3, p2)])
    d2 = layers.Conv2D(base_filters*2, 3, padding='same', activation='relu')(d2)
    if use_cbam:
        d2 = CBAM()(d2)

    d1 = layers.Concatenate(axis=-1)([p1, upsample_to(d2, p1)])
    d1 = layers.Conv2D(base_filters, 3, padding='same', activation='relu')(d1)
    if use_cbam: d1 = CBAM()(d1)

    x_out = layers.Conv2D(base_filters, 3, padding='same', activation='relu')(d1)
    x_out = layers.Conv2D(out_channels, 1, padding='same')(x_out)
    x_out = ResizeToTarget()([x_out, inputs])
    out = layers.Activation('sigmoid')(x_out)

    model = models.Model(inputs=inputs, outputs=out, name='SwinUNet_TF_CBAM')
    return model


In [ ]:
# -------------------------
# helper metrics (numpy)
# -------------------------
def batch_dice_iou_numpy(y_true_tf, y_pred_tf, threshold=0.5, eps=1e-6):
    y_true = y_true_tf.numpy().astype(np.uint8)
    y_pred = y_pred_tf.numpy()
    B = y_true.shape[0]
    dice_list = []
    iou_list = []
    for i in range(B):
        gt = (y_true[i,...,0] > 0).astype(np.uint8).ravel()
        pred = (y_pred[i,...,0] >= threshold).astype(np.uint8).ravel()
        inter = np.sum(gt * pred)
        union = np.sum(gt) + np.sum(pred)
        dice = (2.*inter + eps) / (union + eps)
        union_or = np.sum(np.logical_or(gt>0, pred>0).astype(np.uint8))
        iou = (inter + eps) / (union_or + eps) if union_or > 0 else 0.0
        dice_list.append(dice)
        iou_list.append(iou)
    return float(np.mean(dice_list)), float(np.mean(iou_list))

In [ ]:
# -------------------------
# I/O helpers
# -------------------------
def read_image_mask_paths(img_dir=IMG_DIR, mask_dir=MASK_DIR):
    img_paths = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
    valid_images, mask_paths = [], []
    for img in img_paths:
        base = os.path.splitext(os.path.basename(img))[0]
        mask_name = base + "_segmentation.png"
        mask_path = os.path.join(mask_dir, mask_name)
        if os.path.exists(mask_path):
            valid_images.append(img)
            mask_paths.append(mask_path)
    return valid_images, mask_paths

images, masks = read_image_mask_paths()
train_imgs, val_imgs, train_masks, val_masks = train_test_split(images, masks, test_size=0.15, random_state=SEED)

In [ ]:
def load_image_mask_numpy(img_path, mask_path, size=INPUT_SIZE):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, size).numpy()
    m = tf.io.read_file(mask_path)
    m = tf.image.decode_png(m, channels=1)
    m = tf.image.convert_image_dtype(m, tf.float32)
    m = tf.image.resize(m, size, method='nearest').numpy()
    m = (m > 0.5).astype(np.uint8)
    return img, m

In [ ]:
def random_augment_crop(img, mask, crop_size=PRETRAIN_CROP):
    H, W = img.shape[:2]
    ys, xs = np.where(mask.squeeze() > 0)
    if len(xs) == 0:
        x0 = random.randint(0, max(0, W - crop_size))
        y0 = random.randint(0, max(0, H - crop_size))
    else:
        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()
        cx = int((x_min + x_max) / 2 + random.uniform(-16, 16))
        cy = int((y_min + y_max) / 2 + random.uniform(-16, 16))
        x0 = max(0, min(W - crop_size, cx - crop_size // 2))
        y0 = max(0, min(H - crop_size, cy - crop_size // 2))
    crop_img = img[y0:y0+crop_size, x0:x0+crop_size].copy()
    crop_img = cv2.resize(crop_img, (crop_size, crop_size))
    if random.random() < 0.5:
        crop_img = np.fliplr(crop_img)
    if random.random() < 0.5:
        crop_img = cv2.GaussianBlur(crop_img, (3,3), sigmaX=0.5)
    if random.random() < 0.4:
        factor = 1.0 + random.uniform(-0.15, 0.15)
        crop_img = np.clip(crop_img * factor, 0.0, 1.0)
    return crop_img.astype(np.float32)

In [ ]:
def nt_xent_loss(z_i, z_j, temperature=0.1):
    B = tf.shape(z_i)[0]
    z = tf.concat([z_i, z_j], axis=0)
    z = tf.math.l2_normalize(z, axis=1)
    sim = tf.matmul(z, z, transpose_b=True)
    sim = sim / temperature
    large_neg = -1e9 * tf.eye(tf.shape(sim)[0])
    logits = sim + large_neg
    positives = tf.concat([tf.range(B, 2*B), tf.range(0, B)], axis=0)
    positives = tf.cast(positives, tf.int32)
    loss = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=positives, logits=logits)
    return tf.reduce_mean(loss)

In [ ]:
def build_contrastive_model_from_swin(build_swin_model_fn, input_shape=(PRETRAIN_CROP, PRETRAIN_CROP, 3), proj_dim=128):
    full = build_swin_model_fn(input_shape=input_shape, out_channels=1)
    layer = None
    for name in ['basic_layer_4','basic_layer_3','basic_layer_2','basic_layer_1']:
        try:
            layer = full.get_layer(name)
            break
        except Exception:
            layer = None
    if layer is None:
        feats_out = full.layers[-2].output
    else:
        feats_out = layer.output
    if len(feats_out.shape) == 4:
        pooled = layers.GlobalAveragePooling2D()(feats_out)
    else:
        pooled = feats_out
    proj1 = layers.Dense(512, activation='relu')(pooled)
    proj2 = layers.Dense(proj_dim)(proj1)
    encoder_model = models.Model(inputs=full.input, outputs=proj2, name='swin_encoder_proj')
    return encoder_model, full

In [ ]:
def contrastive_batch_generator(image_paths, mask_paths, batch_size=PRETRAIN_BATCH, crop_size=PRETRAIN_CROP):
    N = len(image_paths)
    idxs = np.arange(N)
    while True:
        np.random.shuffle(idxs)
        for i in range(0, N, batch_size):
            current = idxs[i:i+batch_size]
            if len(current) < batch_size:
                continue
            batch_a = []
            batch_b = []
            for j in current:
                img, m = load_image_mask_numpy(image_paths[j], mask_paths[j], size=INPUT_SIZE)
                a = random_augment_crop(img, m, crop_size=crop_size)
                b = random_augment_crop(img, m, crop_size=crop_size)
                batch_a.append(a)
                batch_b.append(b)
            batch_a = np.stack(batch_a).astype(np.float32)
            batch_b = np.stack(batch_b).astype(np.float32)
            yield batch_a, batch_b


In [ ]:
# -----------------------
# 1) Pretrain contrastive (with progress, save best only)
# -----------------------
def pretrain_contrastive_with_progress(contrastive_model, train_image_paths, train_mask_paths,
                                       batch_size=32, epochs=10, crop_size=128, save_prefix="contrastive"):
    opt = optimizers.Adam(learning_rate=1e-4)
    steps = max(1, len(train_image_paths) // batch_size)
    history = {'loss': []}
    gen = contrastive_batch_generator(train_image_paths, train_mask_paths,
                                      batch_size=batch_size, crop_size=crop_size)

    best_loss = float("inf")
    best_h5 = os.path.join(OUTPUT_DIR, f"{save_prefix}_best.h5")

    for epoch in range(epochs):
        prog = Progbar(steps, stateful_metrics=['loss'])
        epoch_loss = 0.0
        t0 = time.time()
        for step in range(steps):
            a, b = next(gen)
            with tf.GradientTape() as tape:
                z_a = contrastive_model(a, training=True)
                z_b = contrastive_model(b, training=True)
                loss = nt_xent_loss(z_a, z_b, temperature=0.1)
            grads = tape.gradient(loss, contrastive_model.trainable_weights)
            opt.apply_gradients(zip(grads, contrastive_model.trainable_weights))
            loss_val = float(loss)
            epoch_loss += loss_val
            prog.update(step+1, [('loss', loss_val)])

        epoch_loss /= steps
        history['loss'].append(epoch_loss)

        print(f"Epoch {epoch+1}/{epochs} done in {time.time()-t0:.1f}s — avg_loss: {epoch_loss:.4f}")

        if epoch_loss < best_loss:
            prev = best_loss
            best_loss = epoch_loss
            try:
                if os.path.exists(best_h5):
                    os.remove(best_h5)
                contrastive_model.save(best_h5, include_optimizer=False)
                print(f"✅ New best model (.h5) saved at epoch {epoch+1} — loss {prev:.4f} -> {best_loss:.4f} -> {best_h5}")
            except Exception as e:
                print("[warn] failed to save contrastive .h5:", e)

    with open(os.path.join(OUTPUT_DIR, f"history_{save_prefix}.pkl"), 'wb') as f:
        pickle.dump(history, f)

    return history

In [ ]:
# -----------------------
# Frequency/gradient helpers & losses
# -----------------------
def gaussian_blur_kernel(k=5, sigma=1.0):
    ax = np.arange(-k//2 + 1., k//2 + 1.)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx**2 + yy**2) / (2. * sigma**2))
    kernel = kernel / np.sum(kernel)
    kernel = kernel.astype(np.float32)
    kernel = kernel[:, :, None, None]
    return tf.constant(kernel)

_gauss_3 = gaussian_blur_kernel(k=3, sigma=0.8)

In [ ]:
def high_frequency_map(x):
    x_gray = tf.image.rgb_to_grayscale(x)
    x_padded = tf.pad(x_gray, [[0,0],[1,1],[1,1],[0,0]], mode='REFLECT')
    low = tf.nn.conv2d(x_padded, _gauss_3, strides=1, padding='VALID')
    hf = tf.abs(x_gray - low)
    hf_norm = hf / (tf.reduce_max(hf, axis=[1,2,3], keepdims=True) + 1e-8)
    return hf_norm

In [ ]:
def gradient_magnitude_map(prob):
    prob = tf.cast(prob, tf.float32)
    sobel_x = tf.constant([[1,0,-1],[2,0,-2],[1,0,-1]], dtype=tf.float32)
    sobel_y = tf.transpose(sobel_x)
    sobel_x = sobel_x[:, :, None, None]
    sobel_y = sobel_y[:, :, None, None]
    gx = tf.nn.conv2d(prob, sobel_x, strides=1, padding='SAME')
    gy = tf.nn.conv2d(prob, sobel_y, strides=1, padding='SAME')
    g = tf.sqrt(tf.square(gx) + tf.square(gy) + 1e-8)
    g_norm = g / (tf.reduce_max(g, axis=[1,2,3], keepdims=True) + 1e-8)
    return g_norm

In [ ]:
def fbc_loss_fn(y_true, y_pred, image, weight=0.5):
    hf = high_frequency_map(image)
    grad = gradient_magnitude_map(y_pred)
    loss = tf.reduce_mean(tf.square(hf - grad))
    return weight * loss

In [ ]:
def edges_from_mask(gt_mask):
    sobel_x = tf.constant([[1,0,-1],[2,0,-2],[1,0,-1]], dtype=tf.float32)[:,:,None,None]
    sobel_y = tf.transpose(sobel_x, perm=[1,0,2,3])
    gx = tf.nn.conv2d(tf.cast(gt_mask, tf.float32), sobel_x, strides=1, padding='SAME')
    gy = tf.nn.conv2d(tf.cast(gt_mask, tf.float32), sobel_y, strides=1, padding='SAME')
    g = tf.sqrt(gx*gx + gy*gy + 1e-8)
    g = tf.where(g > 0.01, 1.0, 0.0)
    return g

In [ ]:
def tversky_coef_tf(y_true, y_pred, alpha=0.7, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    TP = tf.reduce_sum(y_true_f * y_pred_f)
    FP = tf.reduce_sum((1 - y_true_f) * y_pred_f)
    FN = tf.reduce_sum(y_true_f * (1 - y_pred_f))
    return (TP + smooth) / (TP + alpha * FN + (1-alpha) * FP + smooth)

In [ ]:
def focal_tversky_loss_tf(y_true, y_pred, alpha=0.7, gamma=0.75):
    t = tversky_coef_tf(y_true, y_pred, alpha=alpha)
    return tf.pow((1.0 - t), gamma)

In [ ]:
def combined_segmentation_loss(y_true, y_pred, image, lambda_fbc=0.5, lambda_edge=0.5):
    bce = tf.keras.losses.BinaryCrossentropy()(y_true, y_pred)
    ft = focal_tversky_loss_tf(y_true, y_pred)
    fbc = fbc_loss_fn(y_true, y_pred, image, weight=lambda_fbc)
    edge_gt = edges_from_mask(y_true)
    pred_edge = gradient_magnitude_map(y_pred)
    edge_bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(edge_gt, pred_edge))
    total = bce * 0.5 + ft * 0.5 + fbc + lambda_edge * edge_bce
    return total

In [ ]:
# Build segmentation model
seg_model = build_swin_unet_tf(input_shape=(INPUT_SIZE[0], INPUT_SIZE[1], 3),
                               out_channels=1,
                               embed_dim=96,
                               depths=(2,2,6,2),
                               num_heads=(3,6,12,24),
                               window_size=7,
                               base_filters=32,
                               use_cbam=True)

In [ ]:
# -----------------------
# Datasets and training scaffolding
# -----------------------
def tf_dataset_from_paths(image_paths, mask_paths, batch_size=BATCH, img_size=INPUT_SIZE, augment=True, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(image_paths))
    def _parse(a,b):
        img = tf.io.read_file(a)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, img_size)
        img = tf.image.convert_image_dtype(img, tf.float32)
        mask = tf.io.read_file(b)
        mask = tf.image.decode_png(mask, channels=1)
        mask = tf.image.resize(mask, img_size, method='nearest')
        mask = tf.cast(mask, tf.float32)
        mask = tf.cast(mask > 0.5, tf.float32)
        if augment:
            if tf.random.uniform(()) > 0.5:
                img = tf.image.flip_left_right(img); mask = tf.image.flip_left_right(mask)
            if tf.random.uniform(()) > 0.5:
                img = tf.image.flip_up_down(img); mask = tf.image.flip_up_down(mask)
        return img, mask
    ds = ds.map(_parse, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

In [ ]:
train_ds = tf_dataset_from_paths(train_imgs, train_masks, batch_size=BATCH, augment=True, shuffle=True)
val_ds = tf_dataset_from_paths(val_imgs, val_masks, batch_size=BATCH, augment=False, shuffle=False)

optimizer = optimizers.Adam(learning_rate=LR)

train_loss_metric = tf.keras.metrics.Mean(name='train_loss')
val_loss_metric = tf.keras.metrics.Mean(name='val_loss')
dice_metric = tf.keras.metrics.Mean(name='dice_metric')
iou_metric = tf.keras.metrics.Mean(name='iou_metric')

In [ ]:
@tf.function
def compute_metrics_and_update(masks, preds):
    preds_thresh = tf.cast(preds >= 0.5, tf.float32)
    intersection = tf.reduce_sum(preds_thresh * masks)
    union = tf.reduce_sum(preds_thresh) + tf.reduce_sum(masks)
    dice = (2. * intersection + 1e-6) / (union + 1e-6)
    iou = tf.reduce_sum(tf.cast(tf.logical_or(preds_thresh > 0, masks > 0), tf.float32))
    iou = (intersection + 1e-6) / (iou + 1e-6)
    return dice, iou

In [ ]:
@tf.function
def train_step(images, masks):
    with tf.GradientTape() as tape:
        logits = seg_model(images, training=True)
        loss = combined_segmentation_loss(masks, logits, images, lambda_fbc=0.6, lambda_edge=0.5)
    grads = tape.gradient(loss, seg_model.trainable_weights)
    optimizer.apply_gradients(zip(grads, seg_model.trainable_weights))
    train_loss_metric.update_state(loss)
    dice, iou = compute_metrics_and_update(masks, logits)
    dice_metric.update_state(dice)
    iou_metric.update_state(iou)
    return loss

In [ ]:

@tf.function
def val_step(images, masks):
    preds = seg_model(images, training=False)
    loss = combined_segmentation_loss(masks, preds, images, lambda_fbc=0.6, lambda_edge=0.5)
    val_loss_metric.update_state(loss)
    dice, iou = compute_metrics_and_update(masks, preds)
    dice_metric.update_state(dice)
    iou_metric.update_state(iou)
    return loss


In [ ]:
# -----------------------
# 2) Finetune segmentation (with progress)
# -----------------------
def finetune_segmentation_with_progress(seg_model, train_ds, val_ds, train_image_list,
                                        val_image_list, epochs=50, batch_size=8, lr=1e-4,
                                        save_ckpt=os.path.join(OUTPUT_DIR,'segmentation_best.h5')):
    opt = optimizers.Adam(learning_rate=lr)

    history = {'train_loss': [], 'val_loss': [], 'val_dice': [], 'val_iou': []}
    best_val_dice = 0.0
    patience = 0
    steps_train = max(1, len(train_image_list) // batch_size)
    steps_val = max(1, len(val_image_list) // batch_size)

    for epoch in range(epochs):
        epoch_t0 = time.time()
        prog_tr = Progbar(steps_train, stateful_metrics=['train_loss'])
        train_loss_acc = 0.0

        # train
        for step, (imgs, msks) in enumerate(train_ds):
            with tf.GradientTape() as tape:
                logits = seg_model(imgs, training=True)
                loss = combined_segmentation_loss(msks, logits, imgs, lambda_fbc=0.6, lambda_edge=0.5)
            grads = tape.gradient(loss, seg_model.trainable_weights)
            opt.apply_gradients(zip(grads, seg_model.trainable_weights))
            loss_val = float(loss)
            train_loss_acc += loss_val
            prog_tr.update(step+1, [('train_loss', loss_val)])

        train_loss_epoch = train_loss_acc / steps_train

        # validation
        prog_val = Progbar(steps_val, stateful_metrics=['val_loss','val_dice','val_iou'])
        val_loss_acc = 0.0
        dice_vals = []
        iou_vals = []
        for step, (imgs, msks) in enumerate(val_ds):
            preds = seg_model(imgs, training=False)
            loss = combined_segmentation_loss(msks, preds, imgs, lambda_fbc=0.6, lambda_edge=0.5)
            lval = float(loss)
            val_loss_acc += lval
            d, i = batch_dice_iou_numpy(msks, preds, threshold=0.5)
            dice_vals.append(d); iou_vals.append(i)
            prog_val.update(step+1, [('val_loss', lval), ('val_dice', d), ('val_iou', i)])

        val_loss_epoch = val_loss_acc / steps_val
        val_dice_epoch = float(np.mean(dice_vals))
        val_iou_epoch = float(np.mean(iou_vals))

        history['train_loss'].append(train_loss_epoch)
        history['val_loss'].append(val_loss_epoch)
        history['val_dice'].append(val_dice_epoch)
        history['val_iou'].append(val_iou_epoch)

        print(f"Epoch {epoch+1}/{epochs} — time: {time.time()-epoch_t0:.1f}s — train_loss: {train_loss_epoch:.4f} val_loss: {val_loss_epoch:.4f} val_dice: {val_dice_epoch:.4f} val_iou: {val_iou_epoch:.4f}")

        if val_dice_epoch > best_val_dice:
            best_val_dice = val_dice_epoch
            seg_model.save(save_ckpt)
            print(f"Saved best checkpoint -> {save_ckpt}")
            patience = 0
        else:
            patience += 1

        if patience >= 20:
            print("Early stopping triggered.")
            break

    seg_model.save(os.path.join(OUTPUT_DIR,'seg_model_final.h5'))
    with open(os.path.join(OUTPUT_DIR,'history_finetune.pkl'), 'wb') as f:
        pickle.dump(history, f)
    return history

In [ ]:
# -----------------------
# Refiner model (U-Net like) and trainer
# -----------------------
def conv_block(x, filters, k=3, s=1):
    x = layers.Conv2D(filters, k, strides=s, padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, k, padding='same', activation='relu')(x)
    return x

def build_refiner(input_shape=(INPUT_SIZE[0], INPUT_SIZE[1], 4), base_filters=32):
    inp = layers.Input(shape=input_shape)

    c1 = conv_block(inp, base_filters)
    p1 = layers.MaxPool2D()(c1)

    c2 = conv_block(p1, base_filters*2)
    p2 = layers.MaxPool2D()(c2)

    c3 = conv_block(p2, base_filters*4)
    p3 = layers.MaxPool2D()(c3)

    c4 = conv_block(p3, base_filters*8)
    p4 = layers.MaxPool2D()(c4)

    b = conv_block(p4, base_filters*16)

    u1 = layers.UpSampling2D()(b)
    u1 = layers.Concatenate()([u1, c4])
    c5 = conv_block(u1, base_filters*8)

    u2 = layers.UpSampling2D()(c5)
    u2 = layers.Concatenate()([u2, c3])
    c6 = conv_block(u2, base_filters*4)

    u3 = layers.UpSampling2D()(c6)
    u3 = layers.Concatenate()([u3, c2])
    c7 = conv_block(u3, base_filters*2)

    u4 = layers.UpSampling2D()(c7)
    u4 = layers.Concatenate()([u4, c1])
    c8 = conv_block(u4, base_filters)

    out = layers.Conv2D(1, 1, activation='sigmoid')(c8)
    return models.Model(inputs=inp, outputs=out, name='refiner_net')

refiner = build_refiner()

In [ ]:
def produce_corrupted_masks(mask_batch, prob_erase=0.2, max_iter=2):
    out = []
    for m in mask_batch:
        m = m.squeeze().astype(np.uint8)
        if random.random() < 0.5:
            k = random.choice([3,5])
            if random.random() < 0.5:
                m = cv2.erode(m, np.ones((k,k), np.uint8), iterations=random.randint(1,max_iter))
            else:
                m = cv2.dilate(m, np.ones((k,k), np.uint8), iterations=random.randint(1,max_iter))
        if random.random() < prob_erase:
            for _ in range(random.randint(1,5)):
                h, w = m.shape
                rx = random.randint(0, w-1); ry = random.randint(0, h-1)
                rw = random.randint(5, max(5, int(w*0.15))); rh = random.randint(5, max(5, int(h*0.15)))
                x0 = max(0, rx - rw//2); y0 = max(0, ry - rh//2)
                m[y0:y0+rh, x0:x0+rw] = 0
        m = m.astype(np.float32)
        m = m + np.random.normal(scale=0.05, size=m.shape)
        m = np.clip(m, 0.0, 1.0)
        out.append(m[...,None])
    return np.stack(out)

In [ ]:
def refiner_dataset_from_paths(img_paths, mask_paths, batch_size=8):
    N = len(img_paths)
    while True:
        idxs = np.random.permutation(N)
        for i in range(0, N, batch_size):
            batch = idxs[i:i+batch_size]
            if len(batch) < batch_size: continue
            imgs = []; masks = []
            for j in batch:
                im, m = load_image_mask_numpy(img_paths[j], mask_paths[j], size=INPUT_SIZE)
                imgs.append(im)
                masks.append(m)
            imgs = np.stack(imgs).astype(np.float32)
            masks = np.stack(masks).astype(np.float32)
            corrupted = produce_corrupted_masks(masks, prob_erase=0.4)
            x_in = np.concatenate([imgs, corrupted], axis=-1).astype(np.float32)
            y = masks.astype(np.float32)
            yield x_in, y

In [ ]:
def train_refiner_with_progress(refiner_model, train_image_paths, train_mask_paths, 
                                epochs=20, batch_size=8, save_prefix="refiner"):
    opt = optimizers.Adam(learning_rate=1e-4)
    steps = max(1, len(train_image_paths) // batch_size)
    history = {'loss': []}
    gen = refiner_dataset_from_paths(train_image_paths, train_mask_paths, batch_size=batch_size)

    best_loss = float("inf")
    best_h5 = os.path.join(OUTPUT_DIR, f"{save_prefix}_best.h5")

    for epoch in range(epochs):
        prog = Progbar(steps, stateful_metrics=['loss'])
        loss_acc = 0.0
        t0 = time.time()

        for step in range(steps):
            x, y = next(gen)
            with tf.GradientTape() as tape:
                pred = refiner_model(x, training=True)
                loss = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y, pred))
            grads = tape.gradient(loss, refiner_model.trainable_weights)
            opt.apply_gradients(zip(grads, refiner_model.trainable_weights))

            lval = float(loss)
            loss_acc += lval
            prog.update(step+1, [('loss', lval)])

        avg = loss_acc / steps
        history['loss'].append(avg)

        if avg < best_loss:
            prev = best_loss
            best_loss = avg
            try:
                if os.path.exists(best_h5):
                    os.remove(best_h5)
                refiner_model.save(best_h5, include_optimizer=False)
                print(f"✅ New best refiner saved (epoch {epoch+1}) — avg_loss {prev:.4f} -> {best_loss:.4f}")
            except Exception as e:
                print("[warn] failed to save best refiner .h5:", e)

        print(f"Refiner Epoch {epoch+1}/{epochs} — {time.time()-t0:.1f}s — avg_loss: {avg:.4f}")

    with open(os.path.join(OUTPUT_DIR, f"history_{save_prefix}.pkl"), 'wb') as f:
        pickle.dump(history, f)

    print(f"Training finished. Best avg loss: {best_loss:.4f}. Best HDF5 at: {best_h5}")
    return history


In [ ]:
# -----------------------
# Postprocess + inference helpers
# -----------------------
def postprocess_mask(mask_prob, threshold=0.5, min_size=100):
    mask = (mask_prob >= threshold).astype(np.uint8)
    labeled, n = ndimage.label(mask)
    if n == 0:
        return mask
    counts = np.bincount(labeled.ravel())
    counts[0] = 0
    largest = counts.argmax()
    out = (labeled == largest).astype(np.uint8)
    if out.sum() < min_size:
        return np.zeros_like(out)
    return out

In [ ]:
def predict_with_tta_and_refiner(model, refiner_model, image_np, tta_transforms=['none','flip_lr','flip_ud'], threshold=0.5):
    H, W = image_np.shape[:2]
    inp = cv2.resize(image_np, INPUT_SIZE)
    batch = []
    for t in tta_transforms:
        if t == 'none':
            batch.append(inp)
        elif t == 'flip_lr':
            batch.append(np.fliplr(inp))
        elif t == 'flip_ud':
            batch.append(np.flipud(inp))
    batch = np.stack(batch).astype(np.float32)
    preds = model.predict(batch, verbose=0)
    corrected = []
    for p, t in zip(preds, tta_transforms):
        if t == 'none': corrected.append(p)
        elif t == 'flip_lr': corrected.append(np.fliplr(p))
        elif t == 'flip_ud': corrected.append(np.flipud(p))
    avg = np.mean(np.stack(corrected), axis=0)
    ref_in = np.concatenate([inp, avg], axis=-1)[None,...].astype(np.float32)
    refined = refiner_model.predict(ref_in)[0,...,0]
    refined_resized = cv2.resize(refined, (W, H))
    processed = postprocess_mask(refined_resized, threshold=threshold, min_size=100)
    return processed, refined_resized

In [ ]:
# -----------------------
# 1) Contrastive Pretraining
# -----------------------
pretrain_best_path = os.path.join(OUTPUT_DIR, "contrastive_best.h5")

if os.path.exists(pretrain_best_path):
    contrastive_model = tf.keras.models.load_model(pretrain_best_path, compile=False)
    print("✅ Loaded best pretrain contrastive model from:", pretrain_best_path)
else:
    print("🚀 Starting contrastive pretrain...")
    encoder_proj, swin_full = build_contrastive_model_from_swin(
        build_swin_unet_tf,
        input_shape=(PRETRAIN_CROP, PRETRAIN_CROP, 3),
        proj_dim=128
    )
    pretrain_history = pretrain_contrastive_with_progress(
        encoder_proj, train_imgs, train_masks,
        batch_size=PRETRAIN_BATCH,
        epochs=PRETRAIN_EPOCHS,
        crop_size=PRETRAIN_CROP
    )
    contrastive_model = encoder_proj  


In [ ]:
# -----------------------
# 2) Finetune Segmentation
# -----------------------
finetune_best_path = os.path.join(OUTPUT_DIR, "segmentation_best.h5")

if os.path.exists(finetune_best_path):
    seg_model = tf.keras.models.load_model(finetune_best_path, compile=False)
    print("✅ Loaded best fine-tuned segmentation model from:", finetune_best_path)
else:
    print("🚀 Starting finetune segmentation...")
    finetune_history = finetune_segmentation_with_progress(
        seg_model, train_ds, val_ds, train_imgs, val_imgs,
        epochs=FINETUNE_EPOCHS,
        batch_size=BATCH,
        lr=LR
    )


In [ ]:
# -----------------------
# 3) Train Refiner
# -----------------------
refiner_best_path = os.path.join(OUTPUT_DIR, "refiner_best.h5")

if os.path.exists(refiner_best_path):
    refiner = tf.keras.models.load_model(refiner_best_path, compile=False)
    print("✅ Loaded best refiner model from:", refiner_best_path)
else:
    print("🚀 Starting refiner training...")
    refiner_history = train_refiner_with_progress(
        refiner, train_imgs, train_masks,
        epochs=REFINER_EPOCHS,
        batch_size=8
    )


In [ ]:
# -----------------------
# 4) Visualization
# -----------------------
OUT_PATH = "predictions_overlay_fixed.png"

def show_sample_predictions_overlay(model, refiner_model,
                                    image_paths, mask_paths,
                                    n=6, threshold=0.5,
                                    cmap='Reds', alpha=0.45,
                                    figsize=(15,9), seed=42,
                                    save_path=OUT_PATH):
    random.seed(seed)
    idxs = random.sample(range(len(image_paths)), min(n, len(image_paths)))
    cols, rows = 4, len(idxs)
    plt.figure(figsize=(figsize[0], figsize[1] * rows / 3))

    for i, idx in enumerate(idxs):
        img_path = image_paths[idx]
        mask_path = mask_paths[idx]

        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            print(f"[warn] couldn't read {img_path}")
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

        gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if gt is None:
            print(f"[warn] couldn't read mask {mask_path}")
            continue
        gt_bin = (gt > 127).astype(np.uint8)

        processed_binary, prob_map = predict_with_tta_and_refiner(
            model, refiner_model, img_rgb,
            tta_transforms=['none','flip_lr','flip_ud'],
            threshold=threshold
        )

        H, W = img_rgb.shape[:2]
        if prob_map.shape != (H, W):
            prob_map = cv2.resize(prob_map, (W, H))
        if processed_binary.shape != (H, W):
            processed_binary = cv2.resize(processed_binary.astype(np.uint8), (W, H))
        pred_bin = (processed_binary > 0).astype(np.uint8)

        ax = plt.subplot(rows, cols, i*cols + 1)
        ax.imshow(img_rgb)
        ax.set_title(f"Original\n{os.path.basename(img_path)}")
        ax.axis('off')

        ax = plt.subplot(rows, cols, i*cols + 2)
        ax.imshow(img_rgb)
        ax.imshow(gt_bin, cmap=cmap, alpha=alpha, vmin=0, vmax=1)
        ax.set_title("Ground Truth Mask")
        ax.axis('off')

        ax = plt.subplot(rows, cols, i*cols + 3)
        ax.imshow(img_rgb)
        ax.imshow(pred_bin, cmap=cmap, alpha=alpha, vmin=0, vmax=1)
        ax.set_title("Predicted Mask")
        ax.axis('off')

        ax = plt.subplot(rows, cols, i*cols + 4)
        im = ax.imshow(prob_map, cmap='viridis', vmin=0, vmax=1)
        ax.set_title("Predicted Probability")
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

    plt.tight_layout()
    try:
        fig = plt.gcf()
        fig.savefig(save_path, dpi=200, bbox_inches='tight')
        print("✅ Saved visualization to:", save_path)
    except Exception as e:
        print("⚠️ Could not save figure:", e)
    plt.show()


In [ ]:
show_sample_predictions_overlay(seg_model, refiner, val_imgs, val_masks,
                                n=6, threshold=0.5, cmap='Reds', alpha=0.45,
                                figsize=(15,9), seed=42, save_path=OUT_PATH)

In [ ]:
HISTORY_GLOB = os.path.join(OUTPUT_DIR, "history_*.pkl")
SMOOTH_WINDOW = 1  # >1 for light smoothing
SAVE_DIR = OUTPUT_DIR
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
def _smooth(x, w=1):
    if w <= 1:
        return np.array(x)
    kernel = np.ones(w) / w
    return np.convolve(np.array(x), kernel, mode='same')

In [ ]:
def load_history_file(path):
    with open(path, 'rb') as f:
        hist = pickle.load(f)
    # keep only supported (list/ndarray/tuple of numbers) and convert to numpy arrays
    filtered = {}
    for k, v in hist.items():
        if k == '__name__':
            filtered[k] = v
            continue
        try:
            arr = np.array(v)
            # only keep 1-D numeric arrays with length > 0
            if arr.size > 0 and arr.ndim == 1 and np.issubdtype(arr.dtype, np.number):
                filtered[k] = arr.astype(float)
        except Exception:
            # skip non-serializable or non-numeric values
            pass
    # ensure name
    base = os.path.splitext(os.path.basename(path))[0]
    filtered['__name__'] = filtered.get('__name__', base)
    return filtered

In [ ]:
def find_histories(glob_pattern=HISTORY_GLOB):
    files = sorted(glob(glob_pattern))
    histories = []
    for p in files:
        try:
            histories.append(load_history_file(p))
        except Exception as e:
            print(f"[warn] failed to load history file {p}: {e}")
    if len(histories) == 0:
        print("[info] no history pickles found at:", glob_pattern)
    return histories


In [ ]:
def best_epoch_and_value(arr, higher_is_better=True):
    """Return (best_epoch_1based, best_val)."""
    if arr is None or len(arr) == 0:
        return None, None
    if higher_is_better:
        idx = int(np.nanargmax(arr))
    else:
        idx = int(np.nanargmin(arr))
    return idx+1, float(arr[idx])

In [ ]:
def infer_metric_direction(key):
    """Return True if higher is better for metric named `key`, False if lower is better."""
    key = key.lower()
    # metrics considered losses -> lower better
    if 'loss' in key or 'error' in key or 'bce' in key or 'mse' in key:
        return False
    # everything else assumed higher is better (accuracy, iou, dice, precision, recall, f1)
    return True

In [ ]:
def plot_history_all_metrics(hist, save_dir=SAVE_DIR, smooth_w=SMOOTH_WINDOW, show_plot=True, max_cols=3):
    name = hist.get('__name__', 'history')
    # keys excluding __name__
    keys = [k for k in hist.keys() if k != '__name__']
    if not keys:
        print(f"[{name}] No numeric metrics to plot.")
        return

    # sort keys: loss first, then val_*, then train_*, then others (stable order)
    keys_sorted = sorted(keys, key=lambda k: (0 if 'loss' in k.lower() else 1,
                                             0 if k.startswith('val_') else 1,
                                             0 if k.startswith('train_') else 1,
                                             k))

    n = len(keys_sorted)
    cols = min(max_cols, n)
    rows = math.ceil(n / cols)
    figsize = (5 * cols, 3.6 * rows)
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)  # flatten for easy indexing

    print(f"\n[{name}] plotting {n} metrics -> {rows}x{cols} grid (saved as {name}_all_metrics.png)")

    for i, k in enumerate(keys_sorted):
        ax = axes[i]
        vals = np.array(hist[k], dtype=float)
        if vals.size == 0:
            ax.text(0.5, 0.5, "empty", ha='center')
            continue
        vals_sm = _smooth(vals, smooth_w)
        ax.plot(vals_sm, label=k)
        ax.set_title(k)
        ax.grid(True)
        ax.set_xlabel("Epoch")
        # set y-label depending on guess
        if 'loss' in k.lower():
            ax.set_ylabel("Loss (lower better)")
        else:
            ax.set_ylabel("Score (higher better)")

        # annotate best epoch & value
        hib = infer_metric_direction(k)
        be, bv = best_epoch_and_value(vals, higher_is_better=hib)
        if be is not None:
            ax.axvline(be-1, linestyle='--', linewidth=0.8, alpha=0.6)
            ax.scatter([be-1], [_smooth([bv], smooth_w)[0]], zorder=10)
            txt = f"best: ep {be}\n{bv:.4f}"
            ax.annotate(txt, xy=(be-1, bv), xytext=(5, 5), textcoords='offset points', fontsize=8)

        ax.legend(loc='best', fontsize='x-small')

    # hide any unused axes
    for j in range(n, rows*cols):
        axes[j].axis('off')

    plt.suptitle(name)
    plt.tight_layout(rect=[0,0,1,0.96])
    save_path = os.path.join(save_dir, f"{name}_all_metrics.png")
    try:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved -> {save_path}")
    except Exception as e:
        print("[warn] could not save plot:", e)

    if show_plot:
        plt.show()
    else:
        plt.close(fig)


In [ ]:
def plot_all_histories(histories=None, glob_pattern=HISTORY_GLOB, save_dir=SAVE_DIR, smooth_w=SMOOTH_WINDOW):
    if histories is None:
        histories = find_histories(glob_pattern)
    if not histories:
        print("[info] no histories to plot")
        return
    for hist in histories:
        plot_history_all_metrics(hist, save_dir=save_dir, smooth_w=smooth_w, show_plot=True)

In [ ]:
# --- comparison helper: compare same metric across multiple histories
def compare_metric_across_histories(histories, metric_name, smooth_w=SMOOTH_WINDOW, save_dir=SAVE_DIR, show_plot=True):
    # collect those histories that contain the metric
    present = [(h.get('__name__','?'), h[metric_name]) for h in histories if metric_name in h]
    if len(present) == 0:
        print(f"[compare] Metric '{metric_name}' not found in any history.")
        return
    plt.figure(figsize=(8,5))
    for name, vals in present:
        vals_sm = _smooth(np.array(vals, dtype=float), smooth_w)
        plt.plot(vals_sm, label=name)
        be, bv = best_epoch_and_value(vals, higher_is_better=infer_metric_direction(metric_name))
        if be is not None:
            plt.scatter([be-1],[bv], s=30)
            plt.annotate(f"{name}: ep{be}\n{bv:.4f}", xy=(be-1,bv), xytext=(5,5), textcoords='offset points', fontsize=8)
    plt.title(f"Comparison: {metric_name}")
    plt.xlabel("Epoch")
    plt.grid(True)
    plt.legend(loc='best', fontsize='small')
    save_path = os.path.join(save_dir, f"compare_{metric_name}.png")
    try:
        plt.tight_layout()
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved comparison -> {save_path}")
    except Exception as e:
        print("[warn] could not save comparison:", e)
    if show_plot:
        plt.show()
    else:
        plt.close()

In [ ]:
# 1) plot every history file with all detected metrics
histories = find_histories(HISTORY_GLOB)
plot_all_histories(histories, glob_pattern=HISTORY_GLOB, save_dir=SAVE_DIR, smooth_w=SMOOTH_WINDOW)

In [ ]:
# 2) optional: compare commonly interesting val metrics across models (if present)
# Change or add metric names you care about
metrics_to_compare = ['val_dice', 'val_iou', 'val_precision', 'val_recall', 'val_f1', 'val_accuracy', 'val_loss']
for m in metrics_to_compare:
    compare_metric_across_histories(histories, m, smooth_w=SMOOTH_WINDOW, save_dir=SAVE_DIR)

In [ ]:
# --- silence TF internals and Keras logs ---
import logging
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'   # hide C++ INFO/WARN
tf.get_logger().setLevel('ERROR')
logging.getLogger('tensorflow').setLevel(logging.ERROR)


In [ ]:
# --- config ---
TTA_TRANSFORMS = ['none', 'flip_lr', 'flip_ud']   # set to ['none'] to speed up
THRESHOLD = 0.5
OUT_CSV = os.path.join(OUTPUT_DIR, "val_metrics_per_image.csv")
VERBOSE = False   # keep False so individual image warnings aren't printed

In [ ]:
# --- sanity checks ---
assert 'seg_model' in globals(), "seg_model not found"
assert 'refiner' in globals(), "refiner not found"
assert len(val_imgs) == len(val_masks), "val_imgs / val_masks mismatch"

In [ ]:
# --- helpers ---
def compute_confusion_from_binary(gt_bin, pred_bin):
    gt = (gt_bin > 0).astype(np.uint8)
    pred = (pred_bin > 0).astype(np.uint8)
    TP = int(np.sum((pred == 1) & (gt == 1)))
    FP = int(np.sum((pred == 1) & (gt == 0)))
    FN = int(np.sum((pred == 0) & (gt == 1)))
    TN = int(np.sum((pred == 0) & (gt == 0)))
    return TP, FP, FN, TN

In [ ]:
def metrics_from_confusion(TP, FP, FN, TN):
    eps = 1e-8
    dice = (2*TP) / (2*TP + FP + FN + eps)
    iou = (TP) / (TP + FP + FN + eps)
    total = TP + FP + FN + TN
    acc = (TP + TN) / (total + eps)
    if TP + FP == 0:
        precision = 1.0 if (TP + FN) == 0 else 0.0
    else:
        precision = TP / (TP + FP)
    if TP + FN == 0:
        recall = 1.0 if (TP + FP) == 0 else 0.0
    else:
        recall = TP / (TP + FN)
    return float(dice), float(iou), float(acc), float(precision), float(recall)

In [ ]:
# --- replace predict_with_tta_and_refiner to force verbose=0 ---
def predict_with_tta_and_refiner_quiet(model, refiner_model, image_np,
                                       tta_transforms=TTA_TRANSFORMS,
                                       threshold=THRESHOLD):
    """
    Returns (processed_binary, refined_resized_prob_map).
    Uses verbose=0 for all model.predict calls so no Keras progress bars appear.
    """
    H, W = image_np.shape[:2]
    inp = cv2.resize(image_np, INPUT_SIZE)
    batch = []
    for t in tta_transforms:
        if t == 'none':
            batch.append(inp)
        elif t == 'flip_lr':
            batch.append(np.fliplr(inp))
        elif t == 'flip_ud':
            batch.append(np.flipud(inp))
        else:
            batch.append(inp)
    batch = np.stack(batch).astype(np.float32)

    # ensure verbose=0 -> no "1/1" progress bars
    preds = model.predict(batch, verbose=0)
    corrected = []
    for p, t in zip(preds, tta_transforms):
        if t == 'none':
            corrected.append(p)
        elif t == 'flip_lr':
            corrected.append(np.fliplr(p))
        elif t == 'flip_ud':
            corrected.append(np.flipud(p))
        else:
            corrected.append(p)
    avg = np.mean(np.stack(corrected), axis=0)

    ref_in = np.concatenate([inp, avg], axis=-1)[None, ...].astype(np.float32)
    refined = refiner_model.predict(ref_in, verbose=0)[0, ..., 0]

    refined_resized = cv2.resize(refined, (W, H))
    processed = postprocess_mask(refined_resized, threshold=threshold, min_size=100)
    return processed, refined_resized

In [ ]:
# --- evaluation loop (only tqdm will show) ---
per_image_metrics = []
agg_TP = agg_FP = agg_FN = agg_TN = 0
skipped = 0

for img_path, mask_path in tqdm(zip(val_imgs, val_masks), total=len(val_imgs), desc="Evaluating", leave=True):
    try:
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            skipped += 1
            if VERBOSE:
                print(f"[warn] couldn't read image {img_path}, skipping")
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

        gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if gt is None:
            skipped += 1
            if VERBOSE:
                print(f"[warn] couldn't read mask {mask_path}, skipping")
            continue
        gt_bin = (gt > 127).astype(np.uint8)

        # quiet predict
        pred_bin_processed, prob_map = predict_with_tta_and_refiner_quiet(seg_model, refiner, img_rgb,
                                                                          tta_transforms=TTA_TRANSFORMS,
                                                                          threshold=THRESHOLD)

        # ensure shapes match
        if pred_bin_processed.shape != gt_bin.shape:
            pred_bin_processed = cv2.resize(pred_bin_processed.astype(np.uint8),
                                            (gt_bin.shape[1], gt_bin.shape[0]),
                                            interpolation=cv2.INTER_NEAREST)
            pred_bin_processed = (pred_bin_processed > 0).astype(np.uint8)

        TP, FP, FN, TN = compute_confusion_from_binary(gt_bin, pred_bin_processed)
        agg_TP += TP; agg_FP += FP; agg_FN += FN; agg_TN += TN

        dice, iou, acc, precision, recall = metrics_from_confusion(TP, FP, FN, TN)
        per_image_metrics.append({
            'image': os.path.basename(img_path),
            'dice': dice,
            'iou': iou,
            'accuracy': acc,
            'precision': precision,
            'recall': recall,
            'TP': TP, 'FP': FP, 'FN': FN, 'TN': TN
        })

    except Exception as e:
        skipped += 1
        if VERBOSE:
            print(f"[error] processing {img_path}: {e}")
        continue


In [ ]:

# --- compute and print summary ---
num_evaluated = len(per_image_metrics)
dice_list = [d['dice'] for d in per_image_metrics]
iou_list = [d['iou'] for d in per_image_metrics]
acc_list = [d['accuracy'] for d in per_image_metrics]
prec_list = [d['precision'] for d in per_image_metrics]
rec_list = [d['recall'] for d in per_image_metrics]

mean_dice = float(np.mean(dice_list)) if dice_list else 0.0
mean_iou = float(np.mean(iou_list)) if iou_list else 0.0
mean_acc = float(np.mean(acc_list)) if acc_list else 0.0
mean_prec = float(np.mean(prec_list)) if prec_list else 0.0
mean_rec = float(np.mean(rec_list)) if rec_list else 0.0

global_dice, global_iou, global_acc, global_prec, global_rec = metrics_from_confusion(agg_TP, agg_FP, agg_FN, agg_TN)

print("\n===== Validation results summary =====")
print(f"Evaluated images: {num_evaluated} / {len(val_imgs)} (skipped: {skipped})")
print(f"Mean per-image -> Dice: {mean_dice:.4f}, IoU: {mean_iou:.4f}, Acc: {mean_acc:.4f}")
print(f"Global (pixel-wise) -> Dice: {global_dice:.4f}, IoU: {global_iou:.4f}, Acc: {global_acc:.4f}")
print("======================================\n")




In [ ]:
# --- save per-image CSV ---
fieldnames = ['image','dice','iou','accuracy','precision','recall','TP','FP','FN','TN']
try:
    with open(OUT_CSV, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in per_image_metrics:
            writer.writerow({k: r.get(k, '') for k in fieldnames})
    if VERBOSE:
        print("Per-image metrics saved to:", OUT_CSV)
except Exception as e:
    if VERBOSE:
        print("[warn] could not save CSV:", e)